In [1]:
!apt-get install -y libpango1.0-dev libcairo2-dev pkg-config python3-dev
!pip install manim

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pkg-config is already the newest version (0.29.2-1ubuntu3).
libcairo2-dev is already the newest version (1.16.0-5ubuntu2.1).
libpango1.0-dev is already the newest version (1.50.6+ds-2ubuntu1).
python3-dev is already the newest version (3.10.6-1~22.04.1).
The following packages were automatically installed and are no longer required:
  libbz2-dev libpkgconf3 libreadline-dev
Use 'apt autoremove' to remove them.
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [4]:
!apt-get install -y texlive texlive-latex-extra texlive-fonts-extra dvipng

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages were automatically installed and are no longer required:
  libbz2-dev libpkgconf3 libreadline-dev
Use 'apt autoremove' to remove them.
The following additional packages will be installed:
  dvisvgm fonts-adf-accanthis fonts-adf-berenis fonts-adf-gillius
  fonts-adf-universalis fonts-cabin fonts-cantarell fonts-comfortaa
  fonts-croscore fonts-crosextra-caladea fonts-crosextra-carlito
  fonts-dejavu-core fonts-dejavu-extra fonts-droid-fallback fonts-ebgaramond
  fonts-ebgaramond-extra fonts-font-awesome fonts-freefont-otf
  fonts-freefont-ttf fonts-gfs-artemisia fonts-gfs-complutum fonts-gfs-didot
  fonts-gfs-neohellenic fonts-gfs-olga fonts-gfs-solomos fonts-go
  fonts-junicode fonts-lato fonts-linuxlibertine fonts-lmodern fonts-lobster
  fonts-lobstertwo fonts-noto-color-emoji fonts-noto-core fonts-noto-mono
  fonts-oflb-asana-math fonts-open-sans fonts-roboto-unhint

In [2]:
from manim import *

class MNISTVisualization(Scene):
    def construct(self):
        # ---------------------------------------------------------
        # PART 1: The Data - Flattening a 28x28 Image
        # ---------------------------------------------------------
        title = Text("1. Flattening the MNIST Image", font_size=40).to_edge(UP)
        self.play(Write(title))

        # Create a symbolic 5x5 grid to represent the 28x28 image
        grid = VGroup(*[
            Square(side_length=0.4, fill_opacity=0.2, color=BLUE)
            for _ in range(25)
        ]).arrange_in_grid(rows=5, cols=5)

        grid_label = Text("28x28 Image", font_size=24).next_to(grid, DOWN)

        self.play(Create(grid), FadeIn(grid_label))
        self.wait(1)

        # Flatten the grid into a 1D column vector
        flattened_nodes = VGroup(*[
            Circle(radius=0.15, fill_opacity=0.8, color=BLUE)
            for _ in range(15) # Show 15 nodes as a proxy for 784
        ]).arrange(DOWN, buff=0.1).shift(LEFT * 4)

        vector_label = Text("784 Input Pixels", font_size=24).next_to(flattened_nodes, DOWN)

        self.play(
            Transform(grid[:15], flattened_nodes),
            FadeOut(grid[15:]),
            Transform(grid_label, vector_label)
        )
        self.wait(1)
        self.play(FadeOut(title))

        # ---------------------------------------------------------
        # PART 2: Neural Network Architecture
        # ---------------------------------------------------------
        title2 = Text("2. Building the Network", font_size=40).to_edge(UP)
        self.play(Write(title2))

        # Define layer sizes for visualization (proxy for 784, 256, 256, 10)
        layer_sizes = [8, 6, 6, 4]
        layer_labels = ["Input (784)", "Hidden 1 (256)", "Hidden 2 (256)", "Output (10)"]
        layers = VGroup()

        # Create nodes for each layer
        for i, size in enumerate(layer_sizes):
            layer = VGroup(*[Circle(radius=0.15, fill_color=WHITE, fill_opacity=0.5) for _ in range(size)])
            layer.arrange(DOWN, buff=0.2)
            layers.add(layer)

        layers.arrange(RIGHT, buff=2).shift(DOWN * 0.5)

        # Position the pre-existing input nodes into the network's input layer
        self.play(Transform(grid[:8], layers[0]), FadeOut(grid[8:15]), FadeOut(grid_label))

        # Draw remaining layers and labels
        layer_text_group = VGroup()
        for i in range(1, len(layer_sizes)):
            self.play(Create(layers[i]), run_time=0.5)

        for i, text in enumerate(layer_labels):
            label = Text(text, font_size=20).next_to(layers[i], UP)
            layer_text_group.add(label)
            self.play(FadeIn(label), run_time=0.3)

        # Draw connecting lines (Weights)
        edges = VGroup()
        for i in range(len(layers) - 1):
            layer_edges = VGroup()
            for n1 in layers[i]:
                for n2 in layers[i+1]:
                    edge = Line(n1.get_center(), n2.get_center(), stroke_width=0.5, stroke_opacity=0.3)
                    layer_edges.add(edge)
            edges.add(layer_edges)
            self.play(Create(layer_edges), run_time=0.8)

        self.wait(1)
        self.play(FadeOut(title2))

        # ---------------------------------------------------------
        # PART 3: The Forward Pass (Math)
        # ---------------------------------------------------------
        title3 = Text("3. The Forward Pass", font_size=40).to_edge(UP)
        self.play(Write(title3))

        # Show the equations from the notebook
        eq1 = MathTex(r"Z_1 = X \cdot W_1 + b_1").scale(0.8)
        eq2 = MathTex(r"A_1 = \text{ReLU}(Z_1)").scale(0.8)
        eq3 = MathTex(r"Z_{\text{out}} = A_2 \cdot W_{\text{out}} + b_{\text{out}}").scale(0.8)

        equations = VGroup(eq1, eq2, eq3).arrange(DOWN, aligned_edge=LEFT).to_corner(UL).shift(DOWN * 1.5)

        # Highlight edges to simulate data flow while equations pop up
        self.play(Write(eq1), edges[0].animate.set_color(YELLOW).set_opacity(0.8))
        self.play(Write(eq2), layers[1].animate.set_color(YELLOW))
        self.play(edges[0].animate.set_color(WHITE).set_opacity(0.3), layers[1].animate.set_color(WHITE))

        self.play(edges[1].animate.set_color(YELLOW).set_opacity(0.8))
        self.play(edges[1].animate.set_color(WHITE).set_opacity(0.3))

        self.play(Write(eq3), edges[2].animate.set_color(YELLOW).set_opacity(0.8))
        self.play(layers[3].animate.set_color(GREEN))

        self.wait(2)
        self.play(FadeOut(title3), FadeOut(equations), FadeOut(edges), FadeOut(layers), FadeOut(layer_text_group), FadeOut(grid[:8]))

        # ---------------------------------------------------------
        # PART 4: The Training Loop (Gradient Tape)
        # ---------------------------------------------------------
        title4 = Text("4. Training Loop (25 Epochs)", font_size=40).to_edge(UP)
        self.play(Write(title4))

        # Display Epoch, Cost, and Accuracy trackers
        epoch_tracker = ValueTracker(1)
        cost_tracker = ValueTracker(202.20)
        acc_tracker = ValueTracker(14.6)

        epoch_text = always_redraw(lambda: Text(f"Epoch: {int(epoch_tracker.get_value())}", font_size=36).shift(UP * 1))
        cost_text = always_redraw(lambda: Text(f"Cost: {cost_tracker.get_value():.2f}", font_size=36, color=RED))
        acc_text = always_redraw(lambda: Text(f"Accuracy: {acc_tracker.get_value():.1f}%", font_size=36, color=GREEN).shift(DOWN * 1))

        self.play(FadeIn(epoch_text), FadeIn(cost_text), FadeIn(acc_text))

        # Animate the values changing over time (simulating the print outputs from the notebook)
        self.play(
            epoch_tracker.animate.set_value(25),
            cost_tracker.animate.set_value(76.20),
            acc_tracker.animate.set_value(96.39), # 9639 out of 10000
            run_time=4,
            rate_func=linear
        )

        self.wait(1)

        final_text = Text("Training Complete!", font_size=48, color=YELLOW).shift(DOWN * 2.5)
        self.play(Write(final_text))
        self.wait(2)

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [5]:
%%manim -qm -v WARNING MNISTVisualization

Manim Community v0.20.1

In [9]:
import subprocess
import subprocess
result = subprocess.run(['find', '/', '-name', '*.mp4', '-not', '-path', '*/proc/*'], capture_output=True, text=True)
print(result.stdout)

/usr/share/texlive/texmf-dist/tex/latex/mwe/example-movie.mp4
/usr/local/lib/python3.12/dist-packages/panel/tests/pane/assets/mp4.mp4
/usr/local/lib/python3.12/dist-packages/gradio/media_assets/videos/a.mp4
/usr/local/lib/python3.12/dist-packages/gradio/media_assets/videos/b.mp4
/usr/local/lib/python3.12/dist-packages/gradio/media_assets/videos/world.mp4
/content/media/videos/content/720p30/MNISTVisualization.mp4
/content/media/videos/content/720p30/partial_movie_files/MNISTVisualization/2538612922_3591909666_3692776034.mp4
/content/media/videos/content/720p30/partial_movie_files/MNISTVisualization/2538612922_102892414_4136202213.mp4
/content/media/videos/content/720p30/partial_movie_files/MNISTVisualization/2538612922_1432596116_133749752.mp4
/content/media/videos/content/720p30/partial_movie_files/MNISTVisualization/2538612922_1287027188_528009642.mp4
/content/media/videos/content/720p30/partial_movie_files/MNISTVisualization/2538612922_3981938368_2658725391.mp4
/content/media/videos

In [10]:
from google.colab import files

files.download('/content/media/videos/content/720p30/MNISTVisualization.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>